# Liu2024 — Honest TWFB-DGFMDM × S-JEPA (late-fusion hybrid)

**Goal.** (1) Reproduce the *honest* version of the paper's 5-step **TWFB-DGFMDM** as a strong Riemannian
branch, and (2) fuse it with a frozen-S-JEPA embedding classifier at the **decision level**. This is the
PRISM idea anchored to the honest TWFB (not the simplified published code).

**Honest vs published.** The released figshare code picks the best filter band by **test** accuracy. Here the
time-window × band view is selected by **inner cross-validation on the training fold only** — never the test
set. Exhaustive search over the 7×19 grid replaces the under-specified "backtracking search"; tangent-space +
PCA stands in for the LTSA step; the classifier is either FgMDM (= DGFMDM) or TangentSpace+LDA.

**Branches** (`CONFIG["branches"]`): `twfb` | `sjepa` | `twfb+sjepa`.
**Fusion** (`CONFIG["fusion"]`): `weighted` (α tuned on inner-CV OOF) | `stacking` (logistic on inner-CV OOF).

**Leakage discipline (strict, nested).** Per OUTER fold: TWFB view selection, tangent reference mean, PCA, both
branch classifiers, and the fusion weight/stacker are all fit using the OUTER-train only (with an INNER CV for
selection + out-of-fold probabilities). Covariances are per-trial (no fitting), so they are precomputed for all
trials safely. The OUTER-test fold is touched exactly once, for scoring.

> Logging/artifacts mirror `liu2024_source_mat_sjepa_prelocal_augmented` (timestamped `print`→`run.log`,
> `RUN_ID` from CONFIG hash, `config.json`, CSV/JSON set). **Edit the single CONFIG cell (or apply a sweep) and re-run.**

# 1. Imports

In [1]:
import os, re, sys, json, time, random, hashlib, builtins, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import signal as sp_signal

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

import mne
mne.set_log_level("WARNING")
from braindecode.models import SignalJEPA_PreLocal

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def resolve_device(cfg_device="auto"):
    if cfg_device != "auto":
        return torch.device(cfg_device)
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

print(f"torch {torch.__version__} | numpy {np.__version__}")

/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0+cu128 | numpy 2.4.3


# 2. CONFIG  *(edit this one cell, or apply a sweep JSON, then re-run)*

In [2]:
CONFIG = {
    # ---- identity ----
    "experiment_name": "B_twfb_fgmdm_full_honest",
    "config_note":     "honest TWFB ceiling: 7x19 grid, in-fold selection, FgMDM",

    # ---- paths ----
    "data_root":    "../../liu2024_data/liu2024_figshare/sourcedata",
    "artifact_dir": "../../artifacts/liu2024_twfb_dgfmdm_sjepa_hybrid",

    # ---- S-JEPA checkpoint ----
    "sjepa_repo_id":         "braindecode/signal-jepa_without-chans",
    "sjepa_checkpoint_path": None,
    "sjepa_pretrained":      True,        # False = random-init control (tests if pretraining matters in the hybrid)

    # ---- dataset ----
    "subjects":   "all",
    "random_state": 2026,
    "seed":         2026,
    "sfreq_raw":    500,
    "sfreq_model":  128,

    # ---- which branches + fusion ----
    "branches": "twfb",                   # 'twfb' | 'sjepa' | 'twfb+sjepa'
    "fusion":   "weighted",               # 'weighted' | 'stacking' (used only when branches='twfb+sjepa')

    # ---- cross-validation ----
    "cv_scheme":      "sjepa_5fold",      # 'sjepa_5fold' | 'liu_repeated_holdout'
    "n_splits":       5,
    "n_repeats":      10,
    "test_size":      0.40,
    "inner_cv_splits": 4,                 # inner CV for view selection + OOF fusion fitting

    # ---- TWFB branch (Riemannian) ----
    "twfb_grid":   "full",                # 'full' (7x19) | 'canonical' (fast: 3 windows x 3 bands)
    "twfb_time_windows": [[0.0,1.0],[0.5,1.5],[1.0,2.0],[1.5,2.5],[2.0,3.0],[2.5,3.5],[3.0,4.0]],
    "twfb_bands": [[8,12],[9,13],[10,14],[11,15],[12,16],[13,17],[14,18],[15,19],[16,20],
                   [17,21],[18,22],[19,23],[20,24],[21,25],[22,26],[23,27],[24,28],[25,29],[26,30]],
    "twfb_canonical_windows": [[0.0,4.0],[0.0,2.0],[2.0,4.0]],
    "twfb_canonical_bands":   [[8,13],[13,30],[8,30]],
    # onset of MI (marker==2) inside the 8 s trial; per-trial marker is used when present.
    "onset_marker_value":   2,
    "onset_fallback_sample": 1000,        # ~2.0 s @ 500 Hz
    "onset_plausible_range": [800, 1300],
    "twfb_warmup_samples":  800,          # 1.6 s filter warm-up (matches the MATLAB), trimmed off
    "twfb_filter_order":    4,
    "twfb_filter_phase":    "zero",       # 'zero' (filtfilt, honest) | 'causal' (one-pass, matches MATLAB)
    "twfb_notch_hz":        None,         # e.g. 50 to mimic the MATLAB mains notch; null = off
    "twfb_notch_q":         30.0,
    "twfb_cov_estimator":   "oas",        # 'scm'|'oas'|'lwf'
    "twfb_metric":          "riemann",    # 'riemann'|'logeuclid'
    "twfb_classifier":      "fgmdm",      # 'fgmdm' (= DGFMDM, faithful) | 'ts_lda' (TangentSpace+PCA+LDA, robust proba)
    "twfb_pca_max_components": 20,        # LTSA stand-in: PCA on tangent vectors (ts_lda path)
    "twfb_select_in_oof":   False,        # re-select view inside each inner OOF fold (slow, most rigorous)

    # ---- S-JEPA branch (frozen embeddings) ----
    "sjepa_mi_window_s":   [1.5, 5.7],
    "sjepa_window_samples": 537,
    "sjepa_bandpass_hz":   [0.5, 40.0],
    "sjepa_cached_embeddings_dir": None,  # optional: load cached sub-XX.npz instead of recomputing
    "embedding_hook":  "feature_encoder",
    "embedding_pool":  "mean",
    "use_pca":         True,
    "sjepa_pca_max_components": 15,
    "sjepa_classifier": "shrinkage_lda",  # 'shrinkage_lda' | 'logistic_l2'
    "logistic_C": 1.0,

    # ---- misc ----
    "device": "auto",
}

# ---- derived constants ----
DATA_ROOT   = Path(CONFIG["data_root"])
SFREQ_RAW   = CONFIG["sfreq_raw"]
SFREQ_MODEL = CONFIG["sfreq_model"]
WINDOW_SAMPLES = CONFIG["sjepa_window_samples"]
USE_TWFB  = CONFIG["branches"] in ("twfb", "twfb+sjepa")
USE_SJEPA = CONFIG["branches"] in ("sjepa", "twfb+sjepa")
print(f"branches={CONFIG['branches']} | fusion={CONFIG['fusion']} | twfb_clf={CONFIG['twfb_classifier']} | grid={CONFIG['twfb_grid']} | cv={CONFIG['cv_scheme']}")

branches=twfb | fusion=weighted | twfb_clf=fgmdm | grid=full | cv=sjepa_5fold


## 2.1 Logging & Artifact Init

In [3]:
# --- Run ID, artifact dir, and timestamped logging tee'd to run.log (mirrors prelocal_augmented) ---
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write(stream, text):
    try:
        stream.write(text)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"
        stream.write(text.encode(enc, errors="replace").decode(enc, errors="replace"))

_ORIG_PRINT = builtins.print
def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " "); end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False); file = kwargs.pop("file", None)
    msg = sep.join(str(a) for a in args)
    lead = len(msg) - len(msg.lstrip("\n")); body = msg[lead:]
    def w(t):
        _safe_write(sys.stdout if file is None else file, t)
    if lead:
        w("\n" * lead); _safe_write(_LOG_FILE_HANDLE, "\n" * lead)
    if body:
        stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {body}"
        w(stamped + end); _safe_write(_LOG_FILE_HANDLE, stamped + end)
    else:
        w(end); _safe_write(_LOG_FILE_HANDLE, end)
    if flush:
        sys.stdout.flush(); _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print

# Reproducibility
SEED = int(CONFIG.get("seed", CONFIG.get("random_state", 2026)))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = resolve_device(CONFIG.get("device", "auto"))
with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("=" * 70)
print(f"Experiment: {CONFIG.get('experiment_name')}")
print(f"Note:       {CONFIG.get('config_note')}")
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Device:     {DEVICE} | seed={SEED}")
print("=" * 70)

[2026-06-17 07:26:52] ======================================================================
[2026-06-17 07:26:52] Experiment: B_twfb_fgmdm_full_honest
[2026-06-17 07:26:52] Note:       honest TWFB ceiling: 7x19 grid, in-fold selection, FgMDM
[2026-06-17 07:26:52] Run ID:     20260617_0726_a443cb6f
[2026-06-17 07:26:52] Artifacts:  ../../artifacts/liu2024_twfb_dgfmdm_sjepa_hybrid/20260617_0726_a443cb6f
[2026-06-17 07:26:52] Device:     cpu | seed=2026
[2026-06-17 07:26:52] ======================================================================


# 3. Liu2024 Channel Constants

In [4]:
SOURCE_EEG_NAMES_30 = [
    "Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4",
    "FT7","FT8","Cz","C3","C4","T3","T4","CPz",
    "CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2",
]
CPZ_IDX      = 17                       # CPz is the source reference -> dropped
EEG_KEEP_IDX = [i for i in range(30) if i != CPZ_IDX]
EEG_NAMES    = [SOURCE_EEG_NAMES_30[i] for i in EEG_KEEP_IDX]
N_CHANS      = len(EEG_KEEP_IDX)        # 29
MARKER_SOURCE_COL = 32                  # 0-based col of marker channel in the 33-col source (==2 at MI onset)

def make_liu_info(sfreq):
    info = mne.create_info(ch_names=EEG_NAMES, sfreq=float(sfreq), ch_types=["eeg"] * N_CHANS)
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

MNE_INFO = make_liu_info(CONFIG["sfreq_model"])
CHS_INFO = MNE_INFO["chs"]
print(f"N channels: {N_CHANS} | first 5: {EEG_NAMES[:5]}")

[2026-06-17 07:26:52] N channels: 29 | first 5: ['Fp1', 'Fp2', 'Fz', 'F3', 'F4']


# 4. Data Loading (shared .mat helpers + S-JEPA window)

In [5]:
def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    return int(m.group(1)) if m else int(re.findall(r"\d+", Path(path).stem)[-1])

def _load_mat_arrays(mat_path):
    """Return raw_data (40,33,4000) float64, y (40,) in {0,1}."""
    from scipy.io import loadmat
    mat = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    raw = mat.get("rawdata", mat.get("data", None)); lab = mat.get("labels", mat.get("label", None))
    if raw is None or lab is None:
        for k, v in mat.items():
            if k.startswith("__"): continue
            if hasattr(v, "_fieldnames"):
                if raw is None and "rawdata" in v._fieldnames: raw = getattr(v, "rawdata")
                if lab is None and "label"   in v._fieldnames: lab = getattr(v, "label")
            elif raw is None and isinstance(v, np.ndarray) and v.ndim == 3:
                raw = v
    raw = np.asarray(raw, dtype=np.float64)
    trial_ax = next(ax for ax, sz in enumerate(raw.shape) if sz == 40)
    raw = np.moveaxis(raw, trial_ax, 0)
    if raw.shape[1] != 33 and raw.shape[2] == 33:
        raw = raw.transpose(0, 2, 1)
    assert raw.shape == (40, 33, 4000), f"shape {raw.shape}"
    y = np.asarray(lab, dtype=int).ravel()
    if set(np.unique(y).tolist()).issubset({1, 2}): y = y - 1
    assert len(y) == 40 and set(np.unique(y).tolist()).issubset({0, 1}), f"labels {np.unique(y)}"
    return raw, y

def preprocess_sjepa(mat_path, cfg):
    """Liu .mat -> (40, 29, window_samples) at sfreq_model, y. MNE avg-ref + resample + bandpass."""
    raw, y = _load_mat_arrays(mat_path)
    eeg = raw[:, EEG_KEEP_IDX, :].astype(np.float64)               # 40 x 29 x 4000
    n_trials, n_ch, n_t = eeg.shape
    fs_raw, fs_mod = cfg["sfreq_raw"], cfg["sfreq_model"]
    cont = eeg.transpose(1, 0, 2).reshape(n_ch, n_trials * n_t) * 1e-6   # channels x (trials*time), Volts
    info = mne.create_info(ch_names=EEG_NAMES, sfreq=float(fs_raw), ch_types=["eeg"] * N_CHANS)
    rm = mne.io.RawArray(cont, info, verbose=False)
    rm.set_eeg_reference("average", projection=False, verbose=False)
    rm.resample(fs_mod, npad="auto", verbose=False)
    bp_lo, bp_hi = cfg["sjepa_bandpass_hz"]
    rm.filter(bp_lo, bp_hi, method="fir", phase="zero", verbose=False)
    data = rm.get_data() * 1e6
    n_tr = int(n_t * fs_mod / fs_raw)
    data = data.reshape(N_CHANS, n_trials, n_tr).transpose(1, 0, 2)      # 40 x 29 x n_tr
    s0 = int(cfg["sjepa_mi_window_s"][0] * fs_mod); s1 = s0 + cfg["sjepa_window_samples"]
    assert s1 <= n_tr, f"window [{s0}:{s1}] > {n_tr}"
    X = data[:, :, s0:s1].astype(np.float32)
    assert X.shape == (40, N_CHANS, cfg["sjepa_window_samples"]) and np.isfinite(X).all()
    return X, y

def find_mat_files(root):
    root = Path(root)
    if not root.exists(): raise FileNotFoundError(f"data root not found: {root}")
    return sorted(root.rglob("*.mat"))

## 4.1 Raw 500 Hz loader + MI onset

In [6]:
# --- Raw 500 Hz loader + per-trial MI onset (marker==2 with fallback) ---
def load_subject_raw(mat_path, cfg):
    """Return eeg500 (40,29,4000) microvolts, onsets (40,) sample indices, y (40,)."""
    raw, y = _load_mat_arrays(mat_path)                       # (40,33,4000), y
    eeg = raw[:, EEG_KEEP_IDX, :].astype(np.float64)          # 40 x 29 x 4000
    marker = raw[:, MARKER_SOURCE_COL, :]                     # 40 x 4000
    lo, hi = cfg["onset_plausible_range"]; fb = cfg["onset_fallback_sample"]; mval = cfg["onset_marker_value"]
    onsets = []
    for t in range(40):
        idx = np.where(np.isclose(marker[t], mval))[0]
        idx = idx[(idx >= lo) & (idx <= hi)]
        onsets.append(int(idx[0]) if len(idx) else int(fb))
    return eeg, np.asarray(onsets, dtype=int), y

print("Raw loader defined.")

[2026-06-17 07:26:52] Raw loader defined.


# 5. Honest TWFB-DGFMDM Branch

In [7]:
# --- Honest TWFB-DGFMDM branch ---------------------------------------------------------
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import FgMDM

def _notch(x, fs, f0, q):
    b, a = sp_signal.iirnotch(f0 / (0.5 * fs), q)
    return sp_signal.filtfilt(b, a, x, axis=-1)

def _bandpass(x, fs, lo, hi, order, phase):
    ny = 0.5 * fs
    b, a = sp_signal.butter(order, [lo / ny, hi / ny], btype="band")
    if phase == "causal":
        return sp_signal.lfilter(b, a, x, axis=-1)
    return sp_signal.filtfilt(b, a, x, axis=-1)

def segment_view(eeg500, onsets, window_s, band, cfg):
    """For one (time-window, band) view: notch+bandpass on a warm-up-padded span, then crop to the window."""
    fs = cfg["sfreq_raw"]; warm = cfg["twfb_warmup_samples"]; (a, b) = window_s; (lo, hi) = band
    win_len = int(round((b - a) * fs)); n_t = eeg500.shape[2]
    out = np.zeros((len(onsets), N_CHANS, win_len), dtype=np.float64)
    for i in range(len(onsets)):
        start = onsets[i] + int(round(a * fs))
        w = warm; s0 = start - w
        if s0 < 0: w = max(start, 0); s0 = 0
        s1 = min(start + win_len, n_t)
        seg = eeg500[i, :, s0:s1]
        if cfg.get("twfb_notch_hz"):
            seg = _notch(seg, fs, cfg["twfb_notch_hz"], cfg["twfb_notch_q"])
        seg = _bandpass(seg, fs, lo, hi, cfg["twfb_filter_order"], cfg["twfb_filter_phase"])
        seg = seg[:, w:w + win_len] if seg.shape[1] >= w + win_len else seg[:, -win_len:]
        if seg.shape[1] < win_len:   # pad if a trial ran short
            seg = np.pad(seg, ((0, 0), (0, win_len - seg.shape[1])), mode="edge")
        out[i] = seg
    return out

def grid_views(cfg):
    if cfg["twfb_grid"] == "canonical":
        return cfg["twfb_canonical_windows"], cfg["twfb_canonical_bands"]
    return cfg["twfb_time_windows"], cfg["twfb_bands"]

def precompute_view_covariances(eeg500, onsets, cfg):
    """Covariance set per view for ALL 40 trials (per-trial op -> leakage-safe). Returns dict[(wi,bi)] -> (40,29,29)."""
    windows, bands = grid_views(cfg)
    est = cfg["twfb_cov_estimator"]; views = {}
    for wi, win in enumerate(windows):
        for bi, bnd in enumerate(bands):
            seg = segment_view(eeg500, onsets, win, bnd, cfg)
            views[(wi, bi)] = Covariances(estimator=est).transform(seg)
    return windows, bands, views

def predict_proba_safe(clf, X):
    if hasattr(clf, "predict_proba"):
        try:
            return clf.predict_proba(X)
        except Exception:
            pass
    d = np.asarray(clf.transform(X), dtype=float); z = -d
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def twfb_fit(cov_tr, y_tr, cfg):
    """Fit the TWFB classifier on a covariance set. 'fgmdm' = DGFMDM; 'ts_lda' = TangentSpace+PCA+LDA."""
    if cfg["twfb_classifier"] == "fgmdm":
        clf = FgMDM(metric=cfg["twfb_metric"]); clf.fit(cov_tr, y_tr)
        return {"kind": "fgmdm", "clf": clf}
    ts = TangentSpace(metric=cfg["twfb_metric"]).fit(cov_tr); Z = ts.transform(cov_tr); pca = None
    if Z.shape[1] > cfg["twfb_pca_max_components"]:
        n = min(cfg["twfb_pca_max_components"], Z.shape[0] - 1)
        pca = PCA(n_components=n, random_state=cfg["random_state"]).fit(Z); Z = pca.transform(Z)
    lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(Z, y_tr)
    return {"kind": "ts_lda", "ts": ts, "pca": pca, "lda": lda}

def twfb_proba(obj, cov):
    if obj["kind"] == "fgmdm":
        return predict_proba_safe(obj["clf"], cov)
    Z = obj["ts"].transform(cov)
    if obj["pca"] is not None: Z = obj["pca"].transform(Z)
    return obj["lda"].predict_proba(Z)

def select_view_infold(views, keys, tr_idx, y_tr, cfg):
    """Choose the (window,band) view that maximizes INNER-CV balanced accuracy on the TRAIN trials only."""
    inner = StratifiedKFold(n_splits=cfg["inner_cv_splits"], shuffle=True, random_state=cfg["random_state"])
    best_key, best_score = keys[0], -1.0
    for k in keys:
        cov = views[k][tr_idx]; scores = []
        for itr, iva in inner.split(cov, y_tr):
            if len(np.unique(y_tr[itr])) < 2: continue
            obj = twfb_fit(cov[itr], y_tr[itr], cfg)
            scores.append(balanced_accuracy_score(y_tr[iva], twfb_proba(obj, cov[iva]).argmax(1)))
        s = float(np.mean(scores)) if scores else -1.0
        if s > best_score: best_score, best_key = s, k
    return best_key, best_score

print("TWFB honest branch defined.")

[2026-06-17 07:26:52] TWFB honest branch defined.


# 6. S-JEPA Model + Branch

In [8]:
NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")

def load_sjepa_model(cfg, n_chans, chs_info, n_times, n_outputs=2):
    """Load SignalJEPA_PreLocal (pretrained or random) and freeze all but spatial_conv + final_layer."""
    kwargs = dict(n_chans=n_chans, chs_info=chs_info, n_times=n_times, n_outputs=n_outputs)
    if not cfg.get("sjepa_pretrained", True):
        print("S-JEPA: RANDOM init (control) -> no pretrained weights loaded")
        model = SignalJEPA_PreLocal(**kwargs)
    elif cfg.get("sjepa_checkpoint_path"):
        print(f"S-JEPA: loading local checkpoint {cfg['sjepa_checkpoint_path']}")
        model = SignalJEPA_PreLocal(**kwargs)
        state = torch.load(cfg["sjepa_checkpoint_path"], map_location="cpu")
        miss, unexp = model.load_state_dict(state, strict=False)
        print(f"  missing={len(miss)} unexpected={len(unexp)}")
    else:
        print(f"S-JEPA: from_pretrained {cfg['sjepa_repo_id']}")
        model = SignalJEPA_PreLocal.from_pretrained(cfg["sjepa_repo_id"], **kwargs, strict=False)
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if any(name.startswith(pf) for pf in NEW_LAYER_PREFIXES):
            p.requires_grad = True
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  params total={tot:,} trainable={tr:,} (spatial_conv+final_layer)")
    return model

In [9]:
import copy

class TrialDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.int64))
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def finetune_spatial_conv(model, X_train, y_train, cfg, device):
    """Fine-tune spatial_conv + final_layer on the train fold only (new-pre-local spirit)."""
    model = copy.deepcopy(model)
    for name, module in model.named_modules():
        if any(name.startswith(pf.rstrip(".")) for pf in NEW_LAYER_PREFIXES):
            if hasattr(module, "reset_parameters"):
                module.reset_parameters()
    model.train()
    opt = optim.Adam([p for p in model.parameters() if p.requires_grad],
                     lr=cfg["finetune_lr"], weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    n = len(X_train); n_val = max(2, int(0.2 * n))
    idx = np.random.permutation(n); vi, ti = idx[:n_val], idx[n_val:]
    dl_tr = torch.utils.data.DataLoader(TrialDataset(X_train[ti], y_train[ti]),
                                        batch_size=cfg["finetune_batch_size"], shuffle=True)
    dl_va = torch.utils.data.DataLoader(TrialDataset(X_train[vi], y_train[vi]),
                                        batch_size=cfg["finetune_batch_size"], shuffle=False)
    best = float("inf"); best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()}); pc = 0
    for _ in range(cfg["finetune_epochs"]):
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        model.eval(); vl = 0.0
        with torch.no_grad():
            for xb, yb in dl_va:
                vl += crit(model(xb.to(device)), yb.to(device)).item()
        vl /= max(len(dl_va), 1)
        if vl < best - 1e-4:
            best = vl; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()}); pc = 0
        else:
            pc += 1
            if pc >= cfg["finetune_patience"]:
                break
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    return model

@torch.no_grad()
def extract_embeddings(model, X, cfg, device):
    """Per-trial rich embedding via forward hook on an intermediate module (NOT the 2-D logits)."""
    model.eval()
    pool = cfg.get("embedding_pool", "mean"); hook_name = cfg.get("embedding_hook", "feature_encoder")
    target = getattr(model, hook_name, None)
    if target is None:
        raise AttributeError(f"no submodule '{hook_name}' to hook")
    cap = {}
    def _hook(m, i, o): cap["z"] = o.detach()
    h = target.register_forward_hook(_hook)
    dl = torch.utils.data.DataLoader(TrialDataset(X, np.zeros(len(X), dtype=np.int64)), batch_size=32, shuffle=False)
    embs = []
    try:
        for xb, _ in dl:
            _ = model(xb.to(device)); z = cap["z"]
            if z.dim() == 2:
                feat = z
            else:
                axis = 2 if hook_name == "spatial_conv" else 1
                if pool == "flatten":   feat = z.flatten(start_dim=1)
                elif pool == "max":     feat = z.max(dim=axis).values.flatten(start_dim=1)
                elif pool == "meanmax": feat = torch.cat([z.mean(dim=axis), z.max(dim=axis).values], dim=-1).flatten(start_dim=1)
                else:                   feat = z.mean(dim=axis).flatten(start_dim=1)
            embs.append(feat.cpu().numpy())
    finally:
        h.remove()
    out = np.concatenate(embs, axis=0).astype(np.float32)
    if not hasattr(extract_embeddings, "_printed"):
        print(f"  [hook={hook_name} pool={pool}] embedding matrix: {out.shape}"); extract_embeddings._printed = True
    return out

In [10]:
# --- S-JEPA branch (frozen embeddings; deterministic) ---------------------------------
def sjepa_embeddings_for_subject(sid, mat_path, cfg, base_model, device):
    """Frozen rich embeddings (40, D). Loads cache if sjepa_cached_embeddings_dir is set, else computes."""
    cache = cfg.get("sjepa_cached_embeddings_dir")
    if cache:
        p = Path(cache) / f"sub-{sid:02d}.npz"
        if p.exists():
            d = np.load(p); return d["X"].astype(np.float32), d["y"].astype(int)
    X, y = preprocess_sjepa(mat_path, cfg)
    frozen = copy.deepcopy(base_model).to(device).eval()
    E = extract_embeddings(frozen, X, cfg, device)
    return E.astype(np.float32), y

def sjepa_fit(emb_tr, y_tr, cfg):
    pca = None; E = emb_tr
    if cfg.get("use_pca", True) and E.shape[1] > cfg["sjepa_pca_max_components"]:
        n = min(cfg["sjepa_pca_max_components"], E.shape[0] - 1)
        pca = PCA(n_components=n, random_state=cfg["random_state"]).fit(E); E = pca.transform(E)
    clf = make_clf(cfg, which="sjepa_classifier").fit(E, y_tr)
    return {"pca": pca, "clf": clf}

def sjepa_proba(obj, emb):
    E = emb if obj["pca"] is None else obj["pca"].transform(emb)
    return obj["clf"].predict_proba(E)

print("S-JEPA branch defined.")

[2026-06-17 07:26:52] S-JEPA branch defined.


# 7. Classifier + Diagnostics

In [11]:
def make_clf(cfg, which="classifier"):
    name = cfg.get(which, "shrinkage_lda")
    if name == "shrinkage_lda":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    if name == "logistic_l2":
        return LogisticRegression(C=cfg.get("logistic_C", 1.0), penalty="l2", solver="lbfgs", max_iter=500)
    raise ValueError(f"unknown classifier {name}")

def collapse_diagnostics(y_pred, n_classes=2):
    counts = np.bincount(np.asarray(y_pred, dtype=int), minlength=n_classes)
    dom = counts.max() / counts.sum() if counts.sum() > 0 else 1.0
    return {"collapse_flag": bool(dom > 0.95), "collapse_ratio": float(dom), "pred_counts": counts.tolist()}

# 8. Fusion + Decorrelation

In [12]:
# --- Fusion (fit on inner-CV out-of-fold probabilities of the OUTER-train) -------------
def fit_fusion(p_tw_oof, p_sj_oof, y, cfg):
    if cfg["fusion"] == "stacking":
        Xs = np.column_stack([p_tw_oof[:, 1], p_sj_oof[:, 1]])
        meta = LogisticRegression(max_iter=500).fit(Xs, y)
        return {"kind": "stacking", "meta": meta}
    best_a, best = 0.5, -1.0
    for a in np.linspace(0.0, 1.0, 21):
        pred = ((a * p_tw_oof[:, 1] + (1 - a) * p_sj_oof[:, 1]) >= 0.5).astype(int)
        s = balanced_accuracy_score(y, pred)
        if s > best: best, best_a = s, a
    return {"kind": "weighted", "alpha": float(best_a)}

def apply_fusion(obj, p_tw, p_sj):
    if obj["kind"] == "stacking":
        return obj["meta"].predict_proba(np.column_stack([p_tw[:, 1], p_sj[:, 1]]))
    a = obj["alpha"]; p1 = a * p_tw[:, 1] + (1 - a) * p_sj[:, 1]
    return np.column_stack([1 - p1, p1])

def branch_decorrelation(y, pred_tw, pred_sj):
    tw = (pred_tw == y); sj = (pred_sj == y)
    return {"both_correct": float(np.mean(tw & sj)), "neither_correct": float(np.mean(~tw & ~sj)),
            "only_twfb_correct": float(np.mean(tw & ~sj)), "only_sjepa_correct": float(np.mean(~tw & sj)),
            "complementarity": float(np.mean((tw & ~sj) | (~tw & sj)))}

print("Fusion + decorrelation helpers defined.")

[2026-06-17 07:26:52] Fusion + decorrelation helpers defined.


# 9. Nested Per-Subject Runner

In [13]:
# --- Nested per-subject runner ---------------------------------------------------------
def make_subject_splits(y, cfg):
    scheme = cfg.get("cv_scheme", "sjepa_5fold")
    if scheme == "sjepa_5fold":
        sp = StratifiedKFold(n_splits=cfg["n_splits"], shuffle=True, random_state=cfg["random_state"])
    elif scheme == "liu_repeated_holdout":
        sp = StratifiedShuffleSplit(n_splits=cfg["n_repeats"], test_size=cfg["test_size"], random_state=cfg["random_state"])
    else:
        raise ValueError(f"unknown cv_scheme {scheme}")
    return list(sp.split(np.zeros(len(y)), y))

def _twfb_oof(views, sel_key, keys, tr_idx, y_tr, cfg):
    """OOF TWFB probabilities on the OUTER-train (for fusion fitting)."""
    inner = StratifiedKFold(n_splits=cfg["inner_cv_splits"], shuffle=True, random_state=cfg["random_state"])
    oof = np.full((len(tr_idx), 2), 0.5)
    cov_full = views[sel_key][tr_idx]
    for itr, iva in inner.split(cov_full, y_tr):
        if len(np.unique(y_tr[itr])) < 2: continue
        key = sel_key
        if cfg.get("twfb_select_in_oof", False):
            key, _ = select_view_infold(views, keys, tr_idx[itr], y_tr[itr], cfg)
        obj = twfb_fit(views[key][tr_idx][itr], y_tr[itr], cfg)
        oof[iva] = twfb_proba(obj, views[key][tr_idx][iva])
    return oof

def _sjepa_oof(emb_tr, y_tr, cfg):
    inner = StratifiedKFold(n_splits=cfg["inner_cv_splits"], shuffle=True, random_state=cfg["random_state"])
    oof = np.full((len(y_tr), 2), 0.5)
    for itr, iva in inner.split(emb_tr, y_tr):
        if len(np.unique(y_tr[itr])) < 2: continue
        obj = sjepa_fit(emb_tr[itr], y_tr[itr], cfg)
        oof[iva] = sjepa_proba(obj, emb_tr[iva])
    return oof

def run_subject(sid, y, cfg, views, keys, emb_all):
    rows = []
    for fold_idx, (tr, te) in enumerate(make_subject_splits(y, cfg)):
        y_tr, y_te = y[tr], y[te]
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
            continue
        rec = {"subject_id": int(sid), "fold_id": int(fold_idx), "cv_scheme": cfg.get("cv_scheme"),
               "branches": cfg["branches"], "fusion": cfg["fusion"], "twfb_classifier": cfg["twfb_classifier"],
               "n_train": int(len(y_tr)), "n_test": int(len(y_te)),
               "y_test_counts": np.bincount(y_te, minlength=2).tolist()}
        p_tw = p_sj = None; sel_key = None
        try:
            if USE_TWFB:
                sel_key, sel_score = select_view_infold(views, keys, tr, y_tr, cfg)
                rec["selected_window_idx"], rec["selected_band_idx"] = int(sel_key[0]), int(sel_key[1])
                rec["selected_inner_score"] = float(sel_score)
                obj_tw = twfb_fit(views[sel_key][tr], y_tr, cfg)
                p_tw = twfb_proba(obj_tw, views[sel_key][te])
            if USE_SJEPA:
                obj_sj = sjepa_fit(emb_all[tr], y_tr, cfg)
                p_sj = sjepa_proba(obj_sj, emb_all[te])

            if cfg["branches"] == "twfb":
                p_final = p_tw
            elif cfg["branches"] == "sjepa":
                p_final = p_sj
            else:
                fuse = fit_fusion(_twfb_oof(views, sel_key, keys, tr, y_tr, cfg),
                                  _sjepa_oof(emb_all[tr], y_tr, cfg), y_tr, cfg)
                rec["fusion_param"] = fuse.get("alpha", "stacking")
                p_final = apply_fusion(fuse, p_tw, p_sj)
                # branch-level metrics + decorrelation (diagnostic)
                dt, ds = p_tw.argmax(1), p_sj.argmax(1)
                rec["twfb_balanced_accuracy"]  = float(balanced_accuracy_score(y_te, dt))
                rec["sjepa_balanced_accuracy"] = float(balanced_accuracy_score(y_te, ds))
                for k, v in branch_decorrelation(y_te, dt, ds).items():
                    rec[f"decorr_{k}"] = v

            y_pred = p_final.argmax(1)
        except Exception as exc:
            print(f"  Sub {sid} fold {fold_idx} ERROR: {exc}")
            y_pred = np.zeros(len(y_te), int)

        cm = confusion_matrix(y_te, y_pred, labels=[0, 1]); cma = np.array(cm)
        diag = collapse_diagnostics(y_pred)
        rec.update({
            "accuracy": float(accuracy_score(y_te, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_te, y_pred)),
            "left_recall": float(cma[0, 0] / cma[0].sum()) if cma[0].sum() else float("nan"),
            "right_recall": float(cma[1, 1] / cma[1].sum()) if cma[1].sum() else float("nan"),
            "confusion_matrix": cm.tolist(), "collapse_flag": diag["collapse_flag"],
            "collapse_ratio": diag["collapse_ratio"], "pred_counts": diag["pred_counts"],
        })
        rows.append(rec)
    return rows

print("Nested subject runner defined.")

[2026-06-17 07:26:52] Nested subject runner defined.


# 10. Run All Subjects

In [14]:
mat_files  = find_mat_files(DATA_ROOT)
all_sids   = sorted({subject_id_from_path(f) for f in mat_files})
SUBJECT_IDS = all_sids if CONFIG["subjects"] == "all" else sorted(int(s) for s in CONFIG["subjects"])
sid_to_path = {subject_id_from_path(f): f for f in mat_files if subject_id_from_path(f) in SUBJECT_IDS}
print(f"Found {len(mat_files)} .mat files | using {len(SUBJECT_IDS)} subjects")

BASE_MODEL = None
if USE_SJEPA:
    BASE_MODEL = load_sjepa_model(CONFIG, N_CHANS, CHS_INFO, WINDOW_SAMPLES).to(DEVICE)

ALL_FOLD_RESULTS, SUBJECT_SUMMARIES = [], []
print("=" * 70)
for sid in SUBJECT_IDS:
    mat_path = sid_to_path.get(sid)
    if mat_path is None:
        print(f"  Sub {sid:02d}: no file, skipping"); continue
    try:
        windows = bands = views = keys = None; emb_all = None; y = None
        if USE_TWFB:
            eeg500, onsets, y = load_subject_raw(mat_path, CONFIG)
            windows, bands, views = precompute_view_covariances(eeg500, onsets, CONFIG)
            keys = [(wi, bi) for wi in range(len(windows)) for bi in range(len(bands))]
        if USE_SJEPA:
            emb_all, y_emb = sjepa_embeddings_for_subject(sid, mat_path, CONFIG, BASE_MODEL, DEVICE)
            y = y_emb if y is None else y
        rows = run_subject(sid, y, CONFIG, views, keys, emb_all)
    except Exception as exc:
        print(f"  Sub {sid:02d}: error — {exc}"); continue
    ALL_FOLD_RESULTS.extend(rows)
    baccs = [r["balanced_accuracy"] for r in rows]; coll = sum(r["collapse_flag"] for r in rows)
    SUBJECT_SUMMARIES.append({"subject_id": sid,
        "mean_accuracy": float(np.mean([r["accuracy"] for r in rows])) if rows else float("nan"),
        "mean_balanced_accuracy": float(np.mean(baccs)) if rows else float("nan"),
        "std_balanced_accuracy": float(np.std(baccs)) if rows else float("nan"),
        "n_folds": len(rows), "n_collapsed_folds": coll})
    extra = ""
    if CONFIG["branches"] == "twfb+sjepa" and rows and "twfb_balanced_accuracy" in rows[0]:
        extra = (f"  [twfb={np.mean([r['twfb_balanced_accuracy'] for r in rows])*100:.1f}"
                 f" sjepa={np.mean([r['sjepa_balanced_accuracy'] for r in rows])*100:.1f}"
                 f" compl={np.mean([r['decorr_complementarity'] for r in rows])*100:.1f}%]")
    print(f"  Sub {sid:02d}: bal_acc={np.mean(baccs)*100:5.1f}% ± {np.std(baccs)*100:4.1f}%  collapse={coll}/{len(rows)}{extra}")
print("=" * 70)
print(f"Done. Total folds: {len(ALL_FOLD_RESULTS)}")

[2026-06-17 07:26:52] Found 50 .mat files | using 50 subjects
[2026-06-17 07:26:52] ======================================================================


KeyboardInterrupt: 

# 11. Aggregate, Save Artifacts, Plots

In [ ]:
# --- Aggregate, save artifacts (mirrors prelocal_augmented naming), plots ---
if not ALL_FOLD_RESULTS:
    raise RuntimeError("No folds completed — check data_root and that the run cell executed.")

fold_df    = pd.DataFrame(ALL_FOLD_RESULTS)
subject_df = pd.DataFrame(SUBJECT_SUMMARIES)
cm_total = np.zeros((2, 2), dtype=int)
for r in ALL_FOLD_RESULTS:
    cm_total += np.array(r["confusion_matrix"])
lr = cm_total[0, 0] / cm_total[0].sum() if cm_total[0].sum() else float("nan")
rr = cm_total[1, 1] / cm_total[1].sum() if cm_total[1].sum() else float("nan")

GLOBAL_METRICS = {
    "experiment_name": CONFIG["experiment_name"], "branches": CONFIG["branches"], "fusion": CONFIG["fusion"],
    "twfb_classifier": CONFIG["twfb_classifier"], "twfb_grid": CONFIG["twfb_grid"],
    "twfb_metric": CONFIG["twfb_metric"], "sjepa_pretrained": CONFIG.get("sjepa_pretrained"),
    "cv_scheme": CONFIG.get("cv_scheme"), "n_subjects": len(SUBJECT_SUMMARIES), "n_folds_total": len(ALL_FOLD_RESULTS),
    "mean_accuracy": float(fold_df["accuracy"].mean()), "std_accuracy": float(fold_df["accuracy"].std()),
    "mean_balanced_accuracy": float(fold_df["balanced_accuracy"].mean()),
    "std_balanced_accuracy": float(fold_df["balanced_accuracy"].std()),
    "n_collapsed_folds": int(fold_df["collapse_flag"].sum()), "collapse_rate": float(fold_df["collapse_flag"].mean()),
    "left_recall_agg": float(lr), "right_recall_agg": float(rr), "confusion_matrix": cm_total.tolist(),
}
if CONFIG["branches"] == "twfb+sjepa" and "twfb_balanced_accuracy" in fold_df.columns:
    GLOBAL_METRICS["twfb_branch_balanced_accuracy"]  = float(fold_df["twfb_balanced_accuracy"].mean())
    GLOBAL_METRICS["sjepa_branch_balanced_accuracy"] = float(fold_df["sjepa_balanced_accuracy"].mean())
    GLOBAL_METRICS["mean_complementarity"]           = float(fold_df["decorr_complementarity"].mean())
    GLOBAL_METRICS["mean_only_sjepa_correct"]        = float(fold_df["decorr_only_sjepa_correct"].mean())

fold_df.to_csv(ARTIFACT_DIR / "fold_level_results.csv", index=False)
subject_df.to_csv(ARTIFACT_DIR / "subject_level_summary.csv", index=False)
with open(ARTIFACT_DIR / "global_metrics.json", "w") as f: json.dump(GLOBAL_METRICS, f, indent=2)
with open(ARTIFACT_DIR / "cv_results.json", "w") as f: json.dump(ALL_FOLD_RESULTS, f, indent=2, default=str)
with open(ARTIFACT_DIR / "subject_metrics.json", "w") as f: json.dump(SUBJECT_SUMMARIES, f, indent=2)
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump({"run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR), "config": CONFIG,
               "n_subjects": len(SUBJECT_SUMMARIES), "n_folds": len(ALL_FOLD_RESULTS)}, f, indent=2, default=str)

print("=" * 70)
print(f"GLOBAL — TWFB×S-JEPA [{CONFIG['branches']} | fusion={CONFIG['fusion']} | twfb={CONFIG['twfb_classifier']}]")
print(f"  Balanced accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']*100:.2f}% ± {GLOBAL_METRICS['std_balanced_accuracy']*100:.2f}%")
print(f"  Accuracy:          {GLOBAL_METRICS['mean_accuracy']*100:.2f}% ± {GLOBAL_METRICS['std_accuracy']*100:.2f}%")
print(f"  Left/Right recall: {lr*100:.1f}% / {rr*100:.1f}%")
if "twfb_branch_balanced_accuracy" in GLOBAL_METRICS:
    print(f"  Branch bal_acc: twfb={GLOBAL_METRICS['twfb_branch_balanced_accuracy']*100:.1f}%  "
          f"sjepa={GLOBAL_METRICS['sjepa_branch_balanced_accuracy']*100:.1f}%  "
          f"complementarity={GLOBAL_METRICS['mean_complementarity']*100:.1f}%")
print(f"  Collapsed folds:   {GLOBAL_METRICS['n_collapsed_folds']}/{len(ALL_FOLD_RESULTS)} ({GLOBAL_METRICS['collapse_rate']*100:.1f}%)")
print(f"  Artifacts: {ARTIFACT_DIR}")
print("=" * 70)

# Plots
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm_total, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred L", "Pred R"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True L", "True R"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_total[i, j]), ha="center", va="center")
ax.set_title(f"CM — {CONFIG['branches']} ({CONFIG['twfb_classifier']})")
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight"); plt.close(fig)

fig, ax = plt.subplots(figsize=(max(8, len(subject_df) * 0.4), 4))
sids = subject_df["subject_id"].values; b = subject_df["mean_balanced_accuracy"].values * 100
ax.bar(sids, b, yerr=subject_df["std_balanced_accuracy"].values * 100, capsize=3, color="steelblue", alpha=0.85)
ax.axhline(50, color="red", ls="--", label="chance"); ax.axhline(np.nanmean(b), color="orange", label=f"mean={np.nanmean(b):.1f}%")
ax.set_xlabel("subject"); ax.set_ylabel("balanced acc (%)"); ax.legend()
ax.set_xticks(sids); ax.set_xticklabels(sids, rotation=90, fontsize=7)
ax.set_title(f"Per-subject — {CONFIG['branches']} ({CONFIG['twfb_classifier']})")
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / "subject_performance.png", dpi=200, bbox_inches="tight"); plt.close(fig)
print("Saved CSV/JSON artifacts + confusion_matrix.png + subject_performance.png")

[2026-06-17 02:03:59] ======================================================================
[2026-06-17 02:03:59] GLOBAL — TWFB×S-JEPA [twfb | fusion=weighted | twfb=fgmdm]
[2026-06-17 02:03:59]   Balanced accuracy: 52.50% ± 17.75%
[2026-06-17 02:03:59]   Accuracy:          52.50% ± 17.75%
[2026-06-17 02:03:59]   Left/Right recall: 52.6% / 52.4%
[2026-06-17 02:03:59]   Collapsed folds:   3/250 (1.2%)
[2026-06-17 02:03:59]   Artifacts: ../../artifacts/liu2024_twfb_dgfmdm_sjepa_hybrid/20260617_0001_a443cb6f
[2026-06-17 02:03:59] ======================================================================
[2026-06-17 02:03:59] Saved CSV/JSON artifacts + confusion_matrix.png + subject_performance.png
